# MicroCLIP — Colab A100 Training

Runs any config from the ablation matrix on a Colab A100.

**Setup:** Runtime → Change runtime type → **A100 GPU**.

**Storage layout (Drive-space friendly):**

| What | Where | Size |
|---|---|---|
| COCO images | local SSD (`/content`) | ~20 GB, never touches Drive |
| Checkpoints while training | local SSD (`runs/`) | written every 500 steps + every epoch |
| Checkpoint mirror | Drive `MyDrive/microclip/runs/<run>/` | `last_a.pt` + `last_b.pt` + `best.pt` ≈ **0.5 GB per run** |
| Tokenizer | Drive `MyDrive/microclip/artifacts/` | ~1 MB |

Why not write checkpoints straight to Drive: the trainer replaces the ~214 MB `last.pt`
40–90 times per run. Files deleted or replaced through the Colab Drive mount end up in
Drive **Trash**, which still counts against quota — a single run can fill a 15 GB Drive.
Instead, a background process copies checkpoints to Drive every `SYNC_EVERY_MIN` minutes,
overwriting two fixed slot files in place (no deletes → nothing goes to Trash). If a
copy is torn by preemption, the other slot still holds the previous good checkpoint.

**Preemption:** re-run the whole notebook. The newest readable Drive slot is restored to
local disk and training auto-resumes from it (you lose at most ~`SYNC_EVERY_MIN` minutes).

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > A100"
name = torch.cuda.get_device_name(0)
print(name, "| bf16:", torch.cuda.is_bf16_supported())
if "A100" not in name:
    print("WARNING: not an A100 — configs assume Ampere+ (bf16 AMP, batch 512)")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

PERSIST = "/content/drive/MyDrive/microclip"
os.makedirs(f"{PERSIST}/runs", exist_ok=True)
os.makedirs(f"{PERSIST}/artifacts/tokenizer", exist_ok=True)

# Budget ~0.5 GB of free Drive per run you want to keep. If Drive is near full,
# empty Drive Trash first (drive.google.com → Trash) — it counts against quota.
!df -h /content/drive | tail -1
!du -sh {PERSIST}

In [ ]:
%cd /content
!git clone https://github.com/umutonuryasar/microclip.git 2>/dev/null || git -C microclip pull
%cd /content/microclip
!pip -q install -e .

# pip -e registers src/ via a .pth file, which a running kernel never re-reads —
# without this, `import microclip` in this notebook fails until a runtime restart.
import sys
if "/content/microclip/src" not in sys.path:
    sys.path.insert(0, "/content/microclip/src")

In [ ]:
import os

# Tokenizer is tiny -> lives on Drive so it is trained once.
if not os.path.islink("artifacts"):
    os.system("rm -rf artifacts")
    os.symlink(f"{PERSIST}/artifacts", "artifacts")

# Checkpoints go to the LOCAL SSD (fast, no Drive Trash churn); the sync process
# below mirrors them to Drive. Undo the old runs/ -> Drive symlink if present.
if os.path.islink("runs"):
    os.unlink("runs")
os.makedirs("runs", exist_ok=True)
print("artifacts/ ->", os.readlink("artifacts"))
print("runs/ is local:", os.path.abspath("runs"))

In [ ]:
%%bash
# COCO 2017 to local SSD (~20 GB unzipped, ~40 GB peak while unzipping train2017).
df -h /content | tail -1
mkdir -p data/coco && cd data/coco
for z in train2017 val2017; do
  if [ ! -d $z ]; then
    wget -q -c http://images.cocodataset.org/zips/$z.zip
    unzip -q $z.zip && rm $z.zip
  fi
done
if [ ! -d annotations ]; then
  wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -q annotations_trainval2017.zip && rm annotations_trainval2017.zip
fi
du -sh *

In [ ]:
import os

# One-off; lands on Drive via the artifacts/ symlink.
if not os.path.exists("artifacts/tokenizer/bpe16k.json"):
    !python scripts/train_tokenizer.py --config configs/base.yml
else:
    print("tokenizer exists — skipping")

In [ ]:
# Sanity check on the real data path (~few min on A100) before burning GPU hours.
# Writes only to local runs/smoke_*; cleaned up afterwards. Skip on later sessions.
!python scripts/smoke_test.py --config configs/base.yml && rm -rf runs/smoke_sigmoid runs/smoke_softmax

In [ ]:
import wandb

wandb.login()

## Train

Pick a config from the ablation matrix:

| Run | Config |
|---|---|
| Main: sigmoid / softmax @ b512 | `configs/sigmoid_b512.yml` / `configs/softmax_b512.yml` |
| Batch-size ablations | `configs/ablations/{sigmoid,softmax}_b{128,256}.yml` |
| Optimizer / LR / init / ViT | `configs/ablations/{optimizer_sgd,lr_constant,init_xavier,init_he,vit_tiny}.yml` |

The next cell restores the newest checkpoint from Drive (if any) and starts the
background Drive sync. Then the train cell resumes from `runs/<run_name>/last.pt` automatically.

In [ ]:
%%writefile /content/ckpt_sync.py
"""Mirror runs/<run>/{last,best}.pt to Drive without filling Drive Trash.

last.pt alternates between two fixed slot files (last_a.pt / last_b.pt) and every
copy overwrites a slot in place — no delete/rename on Drive, so no Trash growth.
A copy torn by preemption only damages one slot; the other stays valid.
"""
import os
import shutil
import sys
import time

local, remote, interval, stop = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]
os.makedirs(remote, exist_ok=True)
slots = [os.path.join(remote, "last_a.pt"), os.path.join(remote, "last_b.pt")]
mtime = lambda p: os.path.getmtime(p) if os.path.exists(p) else 0.0
slot = 0 if mtime(slots[0]) <= mtime(slots[1]) else 1  # overwrite the older slot first
seen = {}


def push(name):
    global slot
    src = os.path.join(local, name)
    if not os.path.exists(src):
        return
    # Open first: the trainer's os.replace() can't swap the file under us mid-copy.
    with open(src, "rb") as fsrc:
        m = os.fstat(fsrc.fileno()).st_mtime
        if seen.get(name) == m:
            return
        dst = slots[slot] if name == "last.pt" else os.path.join(remote, name)
        t0 = time.time()
        with open(dst, "wb") as fdst:  # truncate + rewrite the same Drive file
            shutil.copyfileobj(fsrc, fdst, 16 << 20)
    seen[name] = m
    if name == "last.pt":
        slot ^= 1  # flip only after a successful copy
    print(f"[sync {time.strftime('%H:%M:%S')}] {name} -> {os.path.basename(dst)} "
          f"({time.time() - t0:.0f}s)", flush=True)


while True:
    for _ in range(interval):
        if os.path.exists(stop):
            break
        time.sleep(1)
    for name in ("best.pt", "last.pt"):
        try:
            push(name)
        except Exception as e:  # e.g. Drive quota full: log, keep training locally
            print(f"[sync] {name} FAILED: {e!r}", flush=True)
    if os.path.exists(stop):
        os.remove(stop)
        print("[sync] final sync done, exiting", flush=True)
        break

In [ ]:
import os, shutil, subprocess, torch
from microclip.config import load_config

CONFIG = "configs/sigmoid_b512.yml"
SYNC_EVERY_MIN = 15  # max training time lost to a preemption

RUN = load_config(CONFIG)["run_name"]
LOCAL, REMOTE = f"runs/{RUN}", f"{PERSIST}/runs/{RUN}"
STOP = "/content/ckpt_sync.stop"
os.makedirs(LOCAL, exist_ok=True)
os.makedirs(REMOTE, exist_ok=True)

# --- restore newest readable checkpoint from Drive (fresh VM only) ---
if os.path.exists(f"{LOCAL}/last.pt"):
    print("local last.pt exists — using it")
else:
    best_step, src = -1, None
    for cand in ("last_a.pt", "last_b.pt", "last.pt"):  # last.pt = old symlink layout
        p = f"{REMOTE}/{cand}"
        if not os.path.exists(p):
            continue
        try:
            step = torch.load(p, map_location="cpu", weights_only=False)["global_step"]
        except Exception as e:
            print(f"skipping {cand}: unreadable ({type(e).__name__})")
            continue
        print(f"{cand}: step {step}")
        if step > best_step:
            best_step, src = step, p
    if src:
        shutil.copyfile(src, f"{LOCAL}/last.pt")
        print(f"restored {os.path.basename(src)} -> {LOCAL}/last.pt (step {best_step})")
    else:
        print("no Drive checkpoint — fresh run")
    if os.path.exists(f"{REMOTE}/best.pt") and not os.path.exists(f"{LOCAL}/best.pt"):
        shutil.copyfile(f"{REMOTE}/best.pt", f"{LOCAL}/best.pt")

# --- (re)start background Drive sync ---
if "sync_proc" in globals() and sync_proc.poll() is None:
    sync_proc.terminate(); sync_proc.wait()
if os.path.exists(STOP):
    os.remove(STOP)
sync_proc = subprocess.Popen(
    ["python", "/content/ckpt_sync.py", LOCAL, REMOTE, str(SYNC_EVERY_MIN * 60), STOP],
    stdout=open("/content/ckpt_sync.log", "a"), stderr=subprocess.STDOUT)
print(f"sync pid {sync_proc.pid}: {LOCAL} -> {REMOTE} every {SYNC_EVERY_MIN} min "
      f"(log: /content/ckpt_sync.log)")

In [ ]:
# num_workers=8: config default (4) targets the local dev box; Colab A100 VMs have ~12 vCPUs.
!python scripts/train.py --config $CONFIG --set data.num_workers=8

In [ ]:
# Final sync: push the finished last.pt/best.pt to Drive, then stop the sync process.
# Run this after training ends (or before switching CONFIG). Check the log for FAILED lines.
open(STOP, "w").close()
sync_proc.wait()
!tail -5 /content/ckpt_sync.log
!ls -la {REMOTE}
!df -h /content/drive | tail -1